# KLUE-NER 한국어 개체명 인식 (klue/bert-base 파인튜닝)

문장에서 인명(PS), 지명(LC), 기관(OG), 날짜(DT), 시간(TI), 수량(QT)을 찾는 모델입니다.

- 글자 단위로 달린 레이블을 **오프셋으로 서브워드 토큰 단위로 정렬**
- 특수 토큰과 패딩은 **-100**으로 손실에서 제외하고, 3차원 logits를 펼쳐 **토큰 단위 손실** 계산
- `seqeval`로 **개체 단위 F1** 평가
- 학습 루프와 평가 함수는 PyTorch로 직접 작성

자세한 설명은 [README](README.md)를 참고하세요. 런타임 유형은 **T4 GPU**로 설정하세요.

In [1]:
# 1. 환경 설정과 설정값
!pip install -q seqeval  # 개체 단위 F1을 계산하는 라이브러리

import torch
import transformers
from dataclasses import dataclass

@dataclass
class Config:
    model_name: str = "klue/bert-base"  # KLUE 벤치마크와 함께 공개된 한국어 BERT
    max_length: int = 128
    batch_size: int = 32
    learning_rate: float = 5e-5
    epochs: int = 3
    warmup_ratio: float = 0.1
    val_ratio: float = 0.1
    train_size: int | None = None       # None = 학습 데이터 전체 사용
    seed: int = 42
    save_dir: str = "klue-ner-bert"

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"  # GPU일 때만 AMP(혼합 정밀도) 사용

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# 2. 랜덤 시드 고정
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

In [3]:
# 3. 데이터 불러오기 (KLUE-NER: 글자 단위 tokens와 글자마다의 ner_tags)
import pandas as pd

BASE_URL = "https://huggingface.co/datasets/klue/klue/resolve/main/ner/"
train_df = pd.read_parquet(BASE_URL + "train-00000-of-00001.parquet")
test_df = pd.read_parquet(BASE_URL + "validation-00000-of-00001.parquet")  # KLUE test는 정답이 비공개라 validation을 test로 사용

# 데이터셋의 ner_tags 번호 순서와 같은 태그 목록
LABELS = ["B-DT", "I-DT", "B-LC", "I-LC", "B-OG", "I-OG", "B-PS", "I-PS", "B-QT", "I-QT", "B-TI", "I-TI", "O"]
label2id = {label: i for i, label in enumerate(LABELS)}  # 태그 이름 → 번호
id2label = {i: label for i, label in enumerate(LABELS)}  # 번호 → 태그 이름

In [4]:
# 4. 학습 / 검증 분리
if cfg.train_size is not None:  # (선택) 빠른 실험용으로 학습 데이터 줄이기
    train_df = train_df.sample(n=cfg.train_size, random_state=cfg.seed).reset_index(drop=True)

val_df = train_df.sample(frac=cfg.val_ratio, random_state=cfg.seed)  # 원래 인덱스를 유지한 채 10% 무작위 추출
train_df = train_df.drop(val_df.index).reset_index(drop=True)         # 뽑힌 줄을 학습에서 제외
val_df = val_df.reset_index(drop=True)

In [5]:
# 5. 토크나이저와 레이블 정렬
# 레이블은 글자마다 달려 있고 모델은 서브워드 토큰마다 예측하므로, 글자 레이블을 토큰 레이블로 바꿔야 함
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def align_labels(offsets, char_labels):
    token_labels = []
    for start, end in offsets:
        if start == end:  # 특수 토큰·패딩은 원문에 없어서 (0, 0)
            token_labels.append(-100)  # cross_entropy가 손실 계산에서 제외하는 값
        else:
            token_labels.append(char_labels[start])  # 토큰의 첫 글자 레이블
    return token_labels

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [6]:
# 6. 데이터셋과 데이터 로더
from torch.utils.data import Dataset, DataLoader

class NERDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        texts = ["".join(tokens) for tokens in df["tokens"]]  # 글자 리스트 → 문장
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_offsets_mapping=True,  # 토큰마다 원문에서의 글자 위치 (start, end)
        )
        # 오프셋과 글자 레이블로 문장마다 토큰 레이블 만들기
        self.labels = [
            align_labels(offsets, list(char_labels))
            for offsets, char_labels in zip(self.encodings["offset_mapping"], df["ner_tags"])
        ]
        self.encodings.pop("offset_mapping")  # 모델 입력이 아니라서 제거

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = NERDataset(train_df, tokenizer, cfg.max_length)
val_dataset = NERDataset(val_df, tokenizer, cfg.max_length)
test_dataset = NERDataset(test_df, tokenizer, cfg.max_length)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

In [7]:
# 7. 모델: BERT의 모든 토큰 벡터 위에 분류 층을 얹어 토큰마다 13개 태그 점수를 냄
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    cfg.model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)
model = model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly in

In [8]:
# 8. 옵티마이저와 학습률 스케줄러 (문서 분류와 동일)
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=cfg.learning_rate)
total_steps = len(train_loader) * cfg.epochs        # 전체 스텝 = 에폭당 배치 수 × 에폭 수
warmup_steps = int(total_steps * cfg.warmup_ratio)  # 학습률을 0에서 천천히 올리는 구간
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

In [9]:
# 9. 토큰 단위 손실
import torch.nn.functional as F

def compute_loss(logits, labels):
    batch, seq_len, num_labels = logits.shape                  # (batch, seq_len, 13)
    flat_logits = logits.reshape(batch * seq_len, num_labels)  # (batch × seq_len, 13)
    flat_labels = labels.reshape(batch * seq_len)              # (batch × seq_len,)
    return F.cross_entropy(flat_logits, flat_labels)           # 정답이 -100인 위치는 자동으로 제외

In [10]:
# 10. 예측과 정답을 문장별 태그 이름 리스트로 변환 (seqeval 입력 형식)
def to_tag_sequences(preds, labels):
    # preds, labels: (batch, seq_len). labels가 -100인 위치는 제외
    true_tags, pred_tags = [], []
    for pred_row, label_row in zip(preds.tolist(), labels.tolist()):  # 문장 하나씩
        sent_true, sent_pred = [], []
        for p, l in zip(pred_row, label_row):  # 토큰 하나씩
            if l != -100:
                sent_true.append(id2label[l])
                sent_pred.append(id2label[p])
        true_tags.append(sent_true)
        pred_tags.append(sent_pred)
    return true_tags, pred_tags

In [11]:
# 11. 평가 함수
from seqeval.metrics import f1_score, classification_report

@torch.no_grad()  # 평가 때는 기울기를 계산하지 않음
def evaluate(model, loader):
    model.eval()  # 평가 모드: 드롭아웃 끄기
    total_loss, total_tokens = 0.0, 0
    all_true, all_pred = [], []
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}  # 배치의 텐서를 모두 GPU로
        labels = batch.pop("labels")  # 정답 텐서를 꺼내고 batch에는 모델 입력만 남김
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits  # softmax 전의 점수, (batch, seq_len, 13)
            loss = compute_loss(logits, labels)
        n_tokens = (labels != -100).sum().item()  # 실제로 평가한 토큰 수
        total_loss += loss.item() * n_tokens      # 토큰 평균 손실 × 토큰 수 = 손실 합계
        total_tokens += n_tokens

        true_tags, pred_tags = to_tag_sequences(logits.argmax(dim=-1), labels)
        all_true.extend(true_tags)
        all_pred.extend(pred_tags)
    model.train()  # 학습 모드로 되돌리기
    return total_loss / total_tokens, all_true, all_pred  # (토큰당 평균 손실, 정답 태그들, 예측 태그들)

In [12]:
# 12. 학습 루프
from tqdm.auto import tqdm

# AMP용 스케일러: float16에서 작은 기울기가 0이 되지 않도록 손실을 키웠다가 업데이트 전에 되돌림
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
model.train()

for epoch in range(cfg.epochs):
    progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{cfg.epochs}")
    for step, batch in enumerate(progress):
        batch = {key: value.to(device) for key, value in batch.items()}
        labels = batch.pop("labels")

        optimizer.zero_grad()                    # 1. 이전 스텝의 기울기 초기화
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits       # 2. 순전파
            loss = compute_loss(logits, labels)  # 3. 손실 계산
        scaler.scale(loss).backward()            # 4. 역전파 (기울기 계산)
        scaler.step(optimizer)                   # 5. 가중치 업데이트
        scaler.update()                          #    다음 스텝의 손실 배율 조정
        scheduler.step()                         #    학습률 조정

        if step % 50 == 0:
            progress.set_postfix(loss=f"{loss.item():.4f}")

    # 에폭이 끝날 때마다 검증 데이터로 평가
    val_loss, val_true, val_pred = evaluate(model, val_loader)
    print(f"[epoch {epoch + 1}] val_loss={val_loss:.4f}  val_f1={f1_score(val_true, val_pred):.4f}")

epoch 1/3:   0%|          | 0/591 [00:00<?, ?it/s]

/tmp/ipykernel_2373/3736803566.py:21: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()                         #    학습률 조정


[epoch 1] val_loss=0.0581  val_f1=0.8860


epoch 2/3:   0%|          | 0/591 [00:00<?, ?it/s]

[epoch 2] val_loss=0.0551  val_f1=0.8893


epoch 3/3:   0%|          | 0/591 [00:00<?, ?it/s]

[epoch 3] val_loss=0.0572  val_f1=0.8974


In [13]:
# 13. 최종 테스트 평가와 모델 저장
test_loss, test_true, test_pred = evaluate(model, test_loader)
print(f"test_loss={test_loss:.4f}  test_f1={f1_score(test_true, test_pred):.4f}")
print(classification_report(test_true, test_pred, digits=4))  # 개체 유형별 precision / recall / F1

model.save_pretrained(cfg.save_dir)
tokenizer.save_pretrained(cfg.save_dir)

test_loss=0.0787  test_f1=0.8838
              precision    recall  f1-score   support

          DT     0.8282    0.8784    0.8525      2310
          LC     0.7785    0.8334    0.8050      1645
          OG     0.8201    0.8224    0.8212      2162
          PS     0.9249    0.9337    0.9292      4417
          QT     0.9164    0.9342    0.9252      3146
          TI     0.8897    0.9174    0.9033       545

   micro avg     0.8723    0.8957    0.8838     14225
   macro avg     0.8596    0.8866    0.8728     14225
weighted avg     0.8731    0.8957    0.8841     14225



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('klue-ner-bert/tokenizer_config.json', 'klue-ner-bert/tokenizer.json')

In [14]:
# 14. 새 문장에서 개체 찾기
def extract_entities(text, offsets, tags):
    """토큰별 BIO 태그를 (개체 문자열, 유형) 리스트로 묶음"""
    entities = []
    current = None  # 만들고 있는 개체 [시작 글자, 끝 글자, 유형]
    for (start, end), tag in zip(offsets, tags):
        if start == end:  # 특수 토큰은 건너뜀
            continue
        # B-이거나, 이어 갈 개체가 없거나 유형이 다른 I-이면 새 개체 시작
        if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
            if current:
                entities.append((text[current[0]:current[1]], current[2]))  # 만들던 개체 저장
            current = [start, end, tag[2:]]
        elif tag.startswith("I-"):  # 같은 개체가 이어지면 끝 위치만 늘림
            current[1] = end
        else:  # O: 개체가 끝남
            if current:
                entities.append((text[current[0]:current[1]], current[2]))
            current = None
    if current:  # 문장이 개체로 끝난 경우
        entities.append((text[current[0]:current[1]], current[2]))
    return entities


@torch.no_grad()
def predict(text):
    model.eval()
    inputs = tokenizer(text, truncation=True, max_length=cfg.max_length, return_offsets_mapping=True, return_tensors="pt")
    offsets = inputs.pop("offset_mapping")[0].tolist()
    inputs = inputs.to(device)

    logits = model(**inputs).logits               # (1, seq_len, 13)
    pred_ids = logits[0].argmax(dim=-1).tolist()  # 토큰마다 점수가 가장 높은 번호
    tags = [id2label[i] for i in pred_ids]        # 번호 → 태그 이름

    return extract_entities(text, offsets, tags)

In [15]:
for sentence in [
    "세종대왕은 1443년에 훈민정음을 창제했다.",
    "다음 주 금요일 오후 6시에 한국외국어대학교 도서관에서 다섯 명이 만나기로 했다.",
]:
    print(sentence)
    print("  →", predict(sentence))

세종대왕은 1443년에 훈민정음을 창제했다.
  → [('세종대왕', 'PS'), ('1443년', 'DT')]
다음 주 금요일 오후 6시에 한국외국어대학교 도서관에서 다섯 명이 만나기로 했다.
  → [('다음 주 금요일', 'DT'), ('오후 6시', 'TI'), ('한국외국어대학교 도서관', 'LC'), ('다섯 명', 'QT')]
